# Brazilian E-Commerce Order and Delivery Analysis

## Business question

How do product mix, payment patterns, and delivery performance vary
across recorded Olist orders, and what practical areas merit further
investigation?

This project uses descriptive analysis. Results describe the supplied
records and do not establish causation, profit, or performance across
all Brazilian e-commerce.

## Dataset and attribution

Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Provided by Olist and distributed under
[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).

Original files are retained separately from analysis outputs.
Transformations, exclusions, and limitations are documented below.

### Imports and Paths

In [3]:
from pathlib import Path
from datetime import date
import hashlib
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
ARCHIVE_PATH = (
    PROJECT_ROOT / "data" / "raw" / "olist_brazilian_ecommerce.zip"
)
DOCS_DIR = PROJECT_ROOT / "docs"
HASH_PATH = DOCS_DIR / "source_hashes.csv"

assert RAW_DIR.is_dir() , "Raw data folder missing. Check your notebook location."
assert ARCHIVE_PATH.is_file() , "Original ZIP file missing."
assert HASH_PATH.is_file() , "Run the previous hash-export step first."

print("Source paths found.")

Source paths found.


### Record the downloaded snapshot

In [13]:
RETRIEVED_ON = "2026-10-06"
DATASET_VERSION = "1"

VERSION_UPDATED_ON = ""

assert RETRIEVED_ON, "Enter the actual download date."
date.fromisoformat(RETRIEVED_ON)

assert DATASET_VERSION, "Enter the version selected for your download."

if VERSION_UPDATED_ON:
    date.fromisoformat(VERSION_UPDATED_ON)

SOURCE_URL = (
    "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce"
)
LICENSE_URL = "https://creativecommons.org/licenses/by-nc-sa/4.0/"

### Check the File Inventory

In [14]:
EXPECTED_FILES = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

csv_paths = sorted(RAW_DIR.glob("*.csv"))
actual_files = {path.name for path in csv_paths}

missing_files = EXPECTED_FILES - actual_files
unexpected_files = actual_files - EXPECTED_FILES

assert not missing_files, f"Missing CSV files: {sorted(missing_files)}"
assert not unexpected_files, (
    f"Unexpected CSV files: {sorted(unexpected_files)}"
)

print(f"Verified {len(csv_paths)} expected CSV files.")

Verified 9 expected CSV files.


### Verify hashes and inspect schemas

In [15]:
def sha256_file(path):
    """Calculate a SHA-256 hash without loading the whole file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024) , b""):
            digest.update(chunk)

    return digest.hexdigest()

recorded_hashes = pd.read_csv(
    HASH_PATH ,
    dtype='string' ,
    encoding='utf-8-sig'
)

assert set(recorded_hashes.columns) == {
    "filename", "Algorithm", "Hash"
}
assert not recorded_hashes["filename"].duplicated().any()
assert recorded_hashes["Algorithm"].eq("SHA256").all()

source_paths ={
    path.name: path
    for path in {ARCHIVE_PATH , *csv_paths}
}

assert set(recorded_hashes['filename']) == set(source_paths)

hash_lookup = recorded_hashes.set_index("filename")["Hash"]

file_metadata = []
audit_rows = []

for filename , path in source_paths.items():
    actual_hash = sha256_file(path)
    expected_hash = hash_lookup.loc[filename].lower()

    assert actual_hash == expected_hash , (
        f"Hash mismatch: {filename}. Investigate before continuing."
    )

    columns = (
        pd.read_csv(path , nrows=0 , encoding='utf-8-sig').columns.tolist()
        if path.suffix.lower() == '.csv'
        else None
    )

    columns = (
        pd.read_csv(path, nrows=0, encoding="utf-8-sig").columns.tolist()
        if path.suffix.lower() == ".csv"
        else None
    )

    file_metadata.append({
        "filename": filename,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": actual_hash,
        "columns": columns,
    })

    audit_rows.append({
        "filename": filename,
        "bytes": path.stat().st_size,
        "column_count": len(columns) if columns is not None else None,
        "hash_matches": True,
    })

source_audit = pd.DataFrame(audit_rows)
source_audit["column_count"] = source_audit["column_count"].astype("Int64")

display(source_audit)

,filename,bytes,column_count,hash_matches
0,olist_order_reviews_dataset.csv,14451670,7,True
1,olist_order_items_dataset.csv,15438671,7,True
2,olist_products_dataset.csv,2379446,9,True
3,product_category_name_translation.csv,2613,2,True
4,olist_brazilian_ecommerce.zip,44717580,<NA>,True
5,olist_sellers_dataset.csv,174703,4,True
6,olist_geolocation_dataset.csv,61273883,5,True
7,olist_customers_dataset.csv,9033957,5,True
8,olist_order_payments_dataset.csv,5777138,5,True
9,olist_orders_dataset.csv,17654914,8,True


### Save the provenance manifest

In [16]:
manifest = {
    "dataset_title": "Brazilian E-Commerce Public Dataset by Olist",
    "attribution": "Olist",
    "source_url": SOURCE_URL,
    "dataset_version": DATASET_VERSION,
    "version_updated_on": VERSION_UPDATED_ON or None,
    "retrieved_on": RETRIEVED_ON,
    "license": "CC BY-NC-SA 4.0",
    "license_url": LICENSE_URL,
    "hash_algorithm": "SHA-256",
    "files": file_metadata,
}

manifest_path = DOCS_DIR / "data_manifest.json"

manifest_path.write_text(
    json.dumps(manifest , indent=2 , ensure_ascii=False) ,
    encoding='utf-8'
)

print("Saved provenance metadata to docs/data_manifest.json")

Saved provenance metadata to docs/data_manifest.json
